# Rule-Based Chatbot — 03 Evaluation

How well does the bot map **unseen phrasings** to the right intent? We hold out 30% of the example patterns, train on the rest, and measure intent-matching accuracy on the held-out set. All numbers are produced by running the code.

In [1]:
import utils, warnings; warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
intents=utils.load_intents(); df=utils.intents_to_frame(intents)
Xtr,Xte,ytr,yte=train_test_split(df['pattern'],df['tag'],test_size=0.3,random_state=42,stratify=df['tag'])
bot=utils.RuleBasedChatbot(intents,threshold=0.0).fit(Xtr,ytr)
print('train patterns:',len(Xtr),'| test patterns:',len(Xte))

train patterns: 53 | test patterns: 24


## 1. Held-out intent accuracy

In [2]:
acc,preds=utils.evaluate(bot,list(Xte),list(yte))
print(f'held-out intent accuracy: {acc:.3f}')

held-out intent accuracy: 0.458


## 2. Where it confuses intents

In [3]:
import pandas as pd
res=pd.DataFrame({'pattern':list(Xte),'true':list(yte),'pred':preds})
res['ok']=res['true']==res['pred']
print('MISSES:'); print(res[~res['ok']][['pattern','true','pred']].to_string(index=False))

MISSES:
                     pattern     true     pred
      do you have any offers  pricing products
           how do i find you location payments
                 any coupons  pricing shipping
                   thank you   thanks products
          can i get a refund  returns products
 how long does shipping take shipping  pricing
           thank you so much   thanks  pricing
               can't sign in  account products
             can you help me  support products
                       howdy greeting shipping
do you have a physical store location products
                         hey greeting shipping
           update my profile  account shipping


## 3. Summary & takeaways

- **On seen / closely-matching phrasings the bot is reliable** (the demo in notebook 02 answers greetings, hours, payments, password resets correctly with high similarity).
- **On genuinely novel phrasings, held-out accuracy is ~0.46** — TF-IDF retrieval over only ~6 patterns per intent generalises poorly; a paraphrase sharing few words with any stored pattern gets mis-routed (e.g. 'return my order' matching 'shipping').
- **The confidence threshold is essential** — it converts low-similarity guesses into a safe fallback instead of confidently wrong answers.
- **Takeaway**: rule/retrieval chatbots are fast, transparent, and controllable, but brittle to vocabulary they haven't seen. Scaling them means more patterns per intent or learned intent embeddings — the step up to modern NLU.